Final Project for 4AL3
RealWaste

In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision timm scikit-learn seaborn matplotlib pandas tqdm 


In [ ]:

### 1. Base ###
import os
import numpy as np
import pandas as pd
import random
import time
from tqdm import tqdm  
import glob  

### 2. PyTorch ###
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


#
### 3. Image (Augmentation) ###
from torchvision import transforms, models
from PIL import Image

import timm
from timm import create_model

### 5. Scikit-learn ###
from sklearn.model_selection import StratifiedKFold 
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight


### 6. Visualization ###
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# configs (hyperparameters)
BATCH_SIZE = 64
IMG_SIZE = (224,224)
TEST_SPLIT = 0.1
VAL_SPLIT = 0.1
RANDOM_SEED = 42
DATASET_DIR = "../Dataset/realwaste"
NUMOFAUGMENT = 3
# constant
NUMOFCLASS = 9

# random seed need to be implemented later

In [ ]:
# load the dataset and transform

class Load_Transform_Dataset(Dataset):
    def __init__(self, dir=None, file_paths = [], file_labels=[], transform=None):
        self.image_transform = transform

        self.class_dict = {
            0: 'Cardboard', 1: 'Food Organics',         2: 'Glass',
            3: 'Metal',     4: 'Miscellaneous Trash',   5: 'Paper',
            6: 'Plastic',   7: 'Textile Trash',         8: 'Vegetation'
        }

        self.image_paths = list(file_paths)
        self.image_labels = list(file_labels)

        if dir is not None:
            for label_int, label_name in self.class_dict.items():
                class_folder = os.path.join(dir, label_name)

                if not os.path.exists(class_folder):
                    continue

                for image_name in os.listdir(class_folder):
                    image_path = os.path.join(class_folder, image_name)

                    try:
                        with Image.open(image_path) as image:
                            image.verify()
                    except Exception as e:
                        print(f"Error while loading data {image_path}")
                    self.image_paths.append(image_path)
                    self.image_labels.append(label_int)
            
            print(f"Loaded {len(self.image_paths)} images from {dir}")
    

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]
        label = self.image_labels[index]
        image = Image.open(image_path).convert("RGB")
        if self.image_transform:
            image = self.image_transform(image)
            
        
        return image, label




        
        
            


In [ ]:
# split data
from sklearn.model_selection import train_test_split

def split_data(full_images, labels, validate_size, test_size, random_seed=RANDOM_SEED):

    full_images = np.array(full_images)
    labels = np.array(labels)

    training_images, remaining_images, training_labels, remaining_labels = train_test_split(
        full_images, labels, 
        test_size=(validate_size + test_size),
        stratify=labels,
        random_state=random_seed
    )
    
    val_relative_size = validate_size / (validate_size + test_size)
    
    validate_images, test_images, validate_labels, test_labels = train_test_split(
        remaining_images, remaining_labels, 
        test_size=(1 - val_relative_size),
        stratify=remaining_labels,
        random_state=random_seed
    )

    return training_images.tolist(), training_labels.tolist(), validate_images.tolist(), validate_labels.tolist(), test_images.tolist(), test_labels.tolist()



In [ ]:
# custom dataset
from torch.utils.data import ConcatDataset
# regular transform applied to original training set + validation and test set
regular_transform = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),  
])

# augmentation transform applied to the augmented training data
augmentation_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(degrees=45),
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor()
    ])

# get the fulldata
full_dataset = Load_Transform_Dataset(dir=DATASET_DIR,transform=regular_transform)
# get full image path and image label
all_images_path = full_dataset.image_paths
all_image_label = full_dataset.image_labels

# 
training_images, training_labels, validate_images, validate_labels, test_images, test_labels = split_data(
    all_images_path, all_image_label, VAL_SPLIT, TEST_SPLIT, RANDOM_SEED
)

# getting the augmented training data
augmented_image_paths = training_images * NUMOFAUGMENT
augmented_labels = training_labels * NUMOFAUGMENT


# getting the dataset for origianl training and augmented train set
origianl_train_set = Load_Transform_Dataset(
    file_paths=training_images, file_labels=training_labels, transform=regular_transform
)

augmented_train_set = Load_Transform_Dataset(
    file_paths=augmented_image_paths, file_labels=augmented_labels, transform=augmentation_transform
)

# combining original and augmented training set
training_set = ConcatDataset([origianl_train_set, augmented_train_set])

# getting the dataset for validation and test tran set
validation_set = Load_Transform_Dataset(
    file_paths=validate_images, file_labels=validate_labels, transform = regular_transform
)

test_set = Load_Transform_Dataset(
    file_paths=test_images, file_labels=test_labels, transform = regular_transform
)

train_loader = DataLoader(training_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(validation_set,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)

# Print dataset information
print("Train", end='\n\t')
print(f"Batches: {len(train_loader)}", end='\n\t')
print(f"Images: {len(train_loader.dataset)}")
print("Validation", end='\n\t')
print(f"Batches: {len(val_loader)}", end='\n\t')
print(f"Images: {len(val_loader.dataset)}")
print("Test", end='\n\t')
print(f"Batches: {len(test_loader)}", end='\n\t')
print(f"Images: {len(test_loader.dataset)}")

In [ ]:
# model 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def plot_training_validation_loss_accuracy(train_losses, val_losses, train_accuracy, val_accuracy, iterations, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        
    axes[0].plot(iterations, train_losses, label='Training Loss', marker='o')
    axes[0].plot(iterations, val_losses, label='Validation Loss', marker='x')
    axes[0].set_title(model_name + ': Training/Validation Loss per Epoch', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(iterations, train_accuracy, label='Training Accuracy', marker='o')
    axes[1].plot(iterations, val_accuracy, label='Validation Accuracy', marker='x')
    axes[1].set_title(model_name + ': Training/Validation Accuracy per Epoch', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def training_model(model, training_load, validation_load, criterion, optimizer, epochs):

    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    for epoch in range(epochs):
        # Training phase
        model.train()
        print(f"{epoch} started")
        total_running_loss = 0.0
        correct_predict_train = 0
        num_of_train = 0

        for images, labels in training_load:
            train_images = images.to(device)
            train_labels = labels.to(device)

            # Zero the gradientsz
            optimizer.zero_grad()

            # Forward pass
            outputs = model(train_images)
            loss = criterion(outputs, train_labels)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            total_running_loss += loss.item()

            # Calculate accuracy
            _, predicted = torch.max(outputs, 1)
            num_of_train += train_labels.size(0)
            correct_predict_train += (predicted == train_labels).sum().item()

        train_losses.append(total_running_loss / len(training_load))
        train_accs.append(100 * correct_predict_train / num_of_train)

        # Validation phase
        model.eval()
        total_val_loss = 0.0
        correct_predict_val = 0
        num_of_val = 0

        with torch.no_grad():
            for images, labels in validation_load:
                val_images, val_labels = images.to(device), labels.to(device)

                outputs = model(val_images)
                loss = criterion(outputs, val_labels)

                total_val_loss += loss.item()

                # Calculate accuracy
                _, predicted = torch.max(outputs, 1)
                num_of_val += val_labels.size(0)
                correct_predict_val += (predicted == val_labels).sum().item()

        val_losses.append(total_val_loss / len(validation_load))
        val_accs.append(100 * correct_predict_val / num_of_val)

        # Print epoch summary
        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {train_losses[-1]:.4f}, Train Acc: {train_accs[-1]:.2f}% | "
              f"Val Loss: {val_losses[-1]:.4f}, Val Acc: {val_accs[-1]:.2f}%")

    return train_losses, val_losses, train_accs, val_accs


In [ ]:
# things we need for models
labels = np.array(
    list(origianl_train_set.image_labels) + list(augmented_train_set.image_labels)
)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

In [ ]:
# Inception V3
# not implemented for this progress

In [ ]:
# EfficientNetV2

model_name = "EfficientNetV2_S"
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUMOFCLASS)

model = model.to(device)

weights = torch.tensor(class_weights.copy(), dtype=torch.float32).to(device) 
criterion = nn.CrossEntropyLoss(weight=weights)
learning_rate = 1e-5
c_learning_rate = 1e-4 
weight_decay = 1e-2

for param in model.parameters():
    param.requires_grad = False
    
for param in model.features[-1].parameters():  
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = optim.Adam([
    {'params': model.features[-1].parameters(), 'lr': learning_rate},  
    {'params': model.classifier.parameters(), 'lr': c_learning_rate}  
], weight_decay=weight_decay)

epochs = 15
train_losses, val_losses, train_accs, val_accs = training_model(
    model, train_loader, val_loader, criterion, optimizer, epochs
)

model_save_path = f"./trained_{model_name}_model.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

plot_training_validation_loss_accuracy(train_losses, val_losses, train_accs, val_accs, list(range(1, epochs+1)), model_name)



In [ ]:
# ResNet50

model_name = "ResNet50"
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, NUMOFCLASS)

model = model.to(device)

weights = torch.tensor(class_weights.copy(), dtype=torch.float32).to(device) 
criterion = nn.CrossEntropyLoss(weight=weights)


learning_rate = 1e-5
c_learning_rate = 1e-4  
weight_decay = 1e-2

for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

optimizer = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': learning_rate},
    {'params': model.fc.parameters(), 'lr': c_learning_rate}
], weight_decay=weight_decay)

epochs = 15
train_losses, val_losses, train_accs, val_accs = training_model(
    model, train_loader, val_loader, criterion, optimizer, epochs
)

model_save_path = f"./trained_{model_name}_model.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

plot_training_validation_loss_accuracy(train_losses, val_losses, train_accs, val_accs, list(range(1, epochs+1)), model_name)


In [ ]:
# DenseNet121

model_name = "DenseNet121"
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
model.classifier = nn.Linear(model.classifier.in_features, NUMOFCLASS)

model = model.to(device)

weights = torch.tensor(class_weights.copy(), dtype=torch.float32).to(device) 
criterion = nn.CrossEntropyLoss(weight=weights)
learning_rate = 1e-5
c_learning_rate = 1e-4
weight_decay = 1e-2

for param in model.parameters():
    param.requires_grad = False
    
for param in model.features.denseblock4.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = optim.Adam([
    {'params': model.features.denseblock4.parameters(), 'lr': learning_rate},
    {'params': model.classifier.parameters(), 'lr': c_learning_rate}
], weight_decay=weight_decay)

epochs = 15
train_losses, val_losses, train_accs, val_accs = training_model(
    model, train_loader, val_loader, criterion, optimizer, epochs
)

model_save_path = f"./trained_{model_name}_model.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

plot_training_validation_loss_accuracy(train_losses, val_losses, train_accs, val_accs, list(range(1, epochs+1)), model_name)



In [ ]:
# ConveNeXtV2
model_name = "ConvNeXtV2"
model = create_model("convnextv2_base", pretrained=True)

in_features = model.get_classifier().in_features
model.reset_classifier(NUMOFCLASS)

model = model.to(device)

weights = torch.tensor(class_weights.copy(), dtype=torch.float32).to(device) 
criterion = nn.CrossEntropyLoss(weight=weights)

learning_rate = 1e-5
c_learning_rate = 1e-4
weight_decay = 1e-2

for param in model.parameters():
    param.requires_grad = False
    
for param in model.stages[-1].parameters():
    param.requires_grad = True

for param in model.get_classifier().parameters():
    param.requires_grad = True

optimizer = optim.Adam([
    {'params': model.stages[-1].parameters(), 'lr': learning_rate},  
    {'params': model.get_classifier().parameters(), 'lr': c_learning_rate}  
], weight_decay=1e-2)


epochs = 15
train_losses, val_losses, train_accs, val_accs = training_model(
    model, train_loader, val_loader, criterion, optimizer, epochs
)

model_save_path = f"./trained_{model_name}_model.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

plot_training_validation_loss_accuracy(train_losses, val_losses, train_accs, val_accs, list(range(1, epochs+1)), model_name)


In [ ]:
# k-fold maybe implemetned later 

In [ ]:
# List of class names for labeling
class_names = [
    'Cardboard', 
    'Food Organics', 
    'Glass', 
    'Metal', 
    'Miscellaneous Trash', 
    'Paper', 
    'Plastic', 
    'Textile Trash', 
    'Vegetation'
]

# List of models
model_lists = ["EfficientNetV2_S", "ResNet50", "DenseNet121", "ConvNeXtV2"]


def load_models(model_name):
    model_path = f"./trained_{model_name}_model.pth"
    
    if model_name not in model_lists:
        raise ValueError(f"Unknown model: {model_name}")
    
    print(f"Loading {model_name}")
    
    if model_name == "EfficientNetV2_S":
        model = models.efficientnet_v2_s(weights=None)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUMOFCLASS)
    elif model_name == "ResNet50":
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, NUMOFCLASS)
    elif model_name == "DenseNet121":
        model = models.densenet121(weights=None)
        model.classifier = nn.Linear(model.classifier.in_features, NUMOFCLASS)
    elif model_name == "ConvNeXtV2":
        model = create_model("convnextv2_base", pretrained=False, num_classes=NUMOFCLASS)

    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    
    print(f"Model loaded from {model_path}")
    return model


def eval_models(model_name, test_loader):
    model = load_models(model_name)
    
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            output = model(images)
            _, preds = torch.max(output, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Calculate metrics
    accuracy = np.mean(all_preds == all_labels)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4)
    
    return {
        "model_name": model_name,
        "preds": all_preds,
        "labels": all_labels,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "confusion_matrix": cm,
        "report": report
    }


def plot_confusion_matrices(results):
    num_models = len(results)
    cols = 2
    rows = (num_models + 1) // 2
    
    fig, axes = plt.subplots(rows, cols, figsize=(16, 8 * rows), dpi=150)
    
    if num_models == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for i, (model_name, data) in enumerate(results.items()):
        cm = data["confusion_matrix"]
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
        
        sns.heatmap(cm_normalized, annot=True, fmt='.1f', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names, 
                    ax=axes[i], cbar_kws={'label': 'Percentage (%)'})
        axes[i].set_title(f'{model_name}\nAccuracy: {data["accuracy"]:.2%}', fontsize=14)
        axes[i].set_ylabel('True Label', fontsize=12)
        axes[i].set_xlabel('Predicted Label', fontsize=12)
    
    # Remove empty subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    plt.savefig('confusion_matrices_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Confusion matrices saved as 'confusion_matrices_comparison.png'")


def plot_accuracy_comparison(results):
    models = list(results.keys())
    accuracies = [results[m]["accuracy"] * 100 for m in models]
    
    # Sort by accuracy
    sorted_indices = np.argsort(accuracies)[::-1]
    models = [models[i] for i in sorted_indices]
    accuracies = [accuracies[i] for i in sorted_indices]
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(models, accuracies, color=['blue', 'orange', 'green', 'red'][:len(models)])
    
    # Add value labels on bars
    for bar, acc in zip(bars, accuracies):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                f'{acc:.2f}%', ha='center', va='bottom', fontsize=11)
    
    plt.ylabel('Accuracy (%)', fontsize=12)
    plt.title('Model Accuracy Comparison', fontsize=14)
    plt.ylim(0, 105)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('accuracy_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Accuracy comparison saved as 'accuracy_comparison.png'")


def print_results(results):
    print("\n" + "="*80)
    print("MODEL EVALUATION RESULTS")
    print("="*80)
    
    sorted_results = sorted(results.items(), key=lambda x: x[1]["accuracy"], reverse=True)
    
    print(f"\n{'Rank':<6}{'Model':<20}{'Accuracy':<15}{'Precision':<15}{'Recall':<15}{'F1-Score':<15}{'Samples':<10}")
    print("-"*80)
    
    for rank, (model_name, data) in enumerate(sorted_results, 1):
        total_samples = len(data["labels"])
        acc_str = f"{data['accuracy']*100:.2f}%"
        prec_str = f"{data['precision']*100:.2f}%"
        rec_str = f"{data['recall']*100:.2f}%"
        f1_str = f"{data['f1_score']*100:.2f}%"
        print(f"{rank:<6}{model_name:<20}{acc_str:<15}{prec_str:<15}{rec_str:<15}{f1_str:<15}{total_samples:<10}")
    
    print("\n" + "="*80)
    print("DETAILED CLASSIFICATION REPORTS")
    print("="*80)
    
    for model_name, data in results.items():
        print(f"\nModel: {model_name}")
        print("-"*80)
        print(data["report"])


# Main execution
print("\nStarting evaluation")
print("="*80)

# Evaluate all models
model_results = {}
for model_name in model_lists:
    print(f"\nEvaluating: {model_name}")
    print("-"*80)
    
    try:
        result = eval_models(model_name, test_loader)
        model_results[model_name] = result
        print(f"{model_name} evaluation completed - Accuracy: {result['accuracy']:.4%}")
    except Exception as e:
        print(f"Failed to evaluate {model_name}: {e}")

# Print results
print_results(model_results)

# Generate visualizations
print("\n" + "="*80)
print("Generating visualizations")
print("="*80 + "\n")

plot_confusion_matrices(model_results)
plot_accuracy_comparison(model_results)

print("\n" + "="*80)
print("Evaluation complete")
print("="*80)